# Markov Chains

---
In class today we will be implementing a Markov chain to process sentences

---
## Learning Objectives

1. Students will be able to explain the Markov Chain process
1. Implement a Markov Chain


Markov Chains represent a series of events following the Markov Property: future states are memory-less in that they depend only on the current state. This can be expanded to the idea of variable order Markov models where there is a variable-length memory (eg. 1st order Markov Model). Markov models consist of fully observable states. 

> A common example of this is in predicting the weather: We can clearly see the current weather and would like to predict tomorrow's weather. This is also applicable to biology with one case being CpG islands. 

Our goal today will be to implement a Markov model built from words. For our example text, we will use the classic example of Dr. Seuss because of the repetitive nature of the text.

---
## Train Markov model

For our initial implementation of the Markov Model, we will use the simple example of Dr. Seuss: "One fish two fish red fish blue fish."



In [1]:
def build_markov_model(markov_model, new_text):
    '''
    Function to build or add to a 1st order Markov model given a string of text
    We will store the markov model as a dictionary of dictionaries
    The key in the outer dictionary represents the current state
    and the inner dictionary represents the next state with their contents containing
    the transition probabilities.
    Note: This would be easier to read if we were to build a class representation
           of the model rather than a dictionary of dictionaries, but for simplicitiy
           our implementation will just use this structure.
    
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the moarkov_model

    Returns:
        markov_model (dict of dicts): an updated markov_model
        
    Pseudocode:
        Add artificial states for start and end
        For each word in text:
            Increment markov_model[word][next_word]
        
    '''

    words = new_text.split()
    padded = ['*S*'] + words + ['*E*']

    for i in range(1, len(padded)):
        state = padded[i - 1]
        next_word = padded[i]

        if state not in markov_model:
            markov_model[state] = {}
        markov_model[state][next_word] = markov_model[state].get(next_word,0) +1

    return markov_model


In [3]:
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print(markov_model)

{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


###  Nth order Markov chain
In the above model, each event or word is output from only the previous state with no memory of any prior states. While this is useful in some cases, typical biological applications of Markov chains require higher-order models to accurately capture what we know about a system. For instance, in attempting to identify coding regions of a genome, we know that open reading frames (ORFs) contain codon triplets, and so a third or sixth order Markov chain would better describe these regions. Here you will implement a generalized form of our previous Markov Chain to allow for Nth order chains.


In [4]:
def build_markov_model(markov_model, text, order=1):
    '''
    Function to build or add to a Nth order Markov model given a string of text

    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        new_text (str): a string to build or add to the moarkov_model
        order (int): the number of previous states to consider for the model
        
    Returns:
        markov_model (dict of dicts): an updated/new markov_model
    '''

    if markov_model is None:
        markov_model = {}

    words = text.split()
    padded = ['*S*'] *order + words + ['*E*']

    for i in range(order, len(padded)):
        if order == 1:
            state = padded[i - order]

        else:
            state = tuple(padded[i - order:i])

        next_word = padded[i]

        if state not in markov_model:
            markov_model[state] = {}
        markov_model[state][next_word] = markov_model[state].get(next_word,0) + 1

    return markov_model


In [5]:
markov_model = dict()
text = "one fish two fish red fish blue red fish blue"
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 2},
 ('fish', 'blue'): {'red': 1, '*E*': 1},
 ('blue', 'red'): {'fish': 1}}

## Generate text from Markov Model

Markov models are "generative models". That is, the probability states in the model can be used to generate output following the conditional probabilities in the model.

We will now generate a sequence of text from the Markov model. For this section, I recommend using np.random.choice, which allows for you to provide a probability distribution for drawing the next edge in the chain.

In [ ]:
import numpy as np

def get_next_word(current_word, markov_model, seed=42):
    '''
    Function to randomly move a valid next state given a markov model
    and a current state (word)
    
    Args: 
        current_word (tuple): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

        seed is not used as seed is set in generate_random_text. Arg is kept to maintain the funtion as it was provided in the assignment

    Returns:
        next_word (str): a randomly selected next word based on transition probabilies
        
    Pseudocode:
        Calculate transition probilities for all next states from a given state (counts/sum)
        Randomly draw from these to generate the next state

     
    '''

    next_word_counts = markov_model[current_word]
    candidates = list(next_word_counts.keys())
    counts = list(next_word_counts.values())
    total = sum(counts)
    probabilities = [counts / total for counts in counts]

    next_word = np.random.choice(candidates, p=probabilities)
    return next_word

def generate_random_text(markov_model, seed=42):
    '''
    Function to generate text given a markov model
    
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)

    Returns:
        sentence (str): a randomly generated sequence given the model
        
    Pseudocode:
        Initialize sentence at start state
        Until End State:
            append get_next_word(current_word, markov_model)
        Return sentence
        
    '''
    
    np.random.seed(seed) 

    sample_key = next(iter(markov_model.keys()))

    if isinstance(sample_key, tuple):
        order = len(sample_key)
        current_state = tuple(['*S*'] * order)

    else:
        order = 1
        current_state = '*S*'

    sentence = []
    end = False

    while end == False:
        next_word = get_next_word(current_state, markov_model, seed)

        if next_word == '*E*':
            end = True

        else:
            sentence.append(next_word)

            if order == 1:
                current_state = next_word

            else:
                current_state = tuple(list(current_state[1:]) + [next_word])

    return ' '.join(sentence)

---

## All the Fish
Up till now, you have only been working with a line or two of the Dr. Seuss' _One Fish, Two Fish_. Now, I want you to build a model using the whole book and try different orders of Markov models.

> **Pro-tip**: Consider how you signify the beginning of the book, beginning of a line, end of a line, and end of the book.

In [8]:
# Now just add some more training data to the markov model. You can find it under data/one_fish_two_fish.txt

markov_model = dict()
# Read in the whole book
with open("data/one_fish_two_fish.txt", "r") as f:
    lines = f.readlines()

book = ""

for line in lines:
    line = line.strip()
    if line: # skip blank lines
        book = book + ' ' + line

markov_model = build_markov_model(markov_model, book)

print(generate_random_text(markov_model, seed=7))

One fish, Blue hair grows fast. So have come a Zans. A bird was "How to Cook" We don't know. Go ask your ear can hear, my hand with a Wump with a hook. On my hand I like to drink. He is off, my hand I said Hello. Can you please look in your dad. Some are everywhere.


---
## Pick Your Poison
There are three texts provided for under `data/`. The first is:
1. Dr. Seuss' "One Fist, Two Fish" (179 lines of text)
2. All of Shakespeare's sonnets (2308 lines of text)
3. Homer's "The Odyssey" (9255 Lines of text)

In [ ]:
# An example of a more complex text that we can use to generate more complex output
nth_order_markov_model = dict()
poison_markov_model = dict()

with open("data/sonnets.txt", "r") as poison_text:

    # Process the lines. Consider that sonnets are separated by an empty line.
    lines = poison_text.readlines()

corpus = ""

for line in lines:
    line = line.strip()
    if line: # skip blank lines
        corpus = corpus + ' ' + line
        
nth_order_markov_model = build_markov_model(poison_markov_model, corpus, order=2)

print (generate_random_text(poison_markov_model,seed=7))

From fairest creatures we desire increase, That thereby beauty's rose might never die, But as the marigold at the sun's eye, And in some perfumes is there more delight than hawks and hounds, some in their horse; And every fair from fair sometime declines, By chance, or nature's changing course untrimm'd: But thy eternal summer shall not be so bold, Although I swear it to me: This told, I joy; but then no longer than thy sins are; For to thy pure and most precious jewel. Yet, in good faith, some say that thee behold, Thy face hath not figur'd to thee resort. As on the finger of a worthier pen; Yet what the best is dressing old words new, Spending again what is most evident: For thou not renewest, Thou dost love her, because thou know'st I am old? O! love's best habit is in my mind, when body's work's expired: For then my state, Like to the painted banquet bids my heart; Wound me not by art, Tell me thou lov'st me for some fault, And I will acquaintance strangle, and look another way: So